# Train Lick Classifier

**Kernel: cliqr-network-training**

Trains `LickTransformer` on labeled `.pt` files produced by `LabelingTool.ipynb`.

**Before running:** export labels from `LabelingTool.ipynb` by running its Save cell for each labeled file.

Splits: **train** (gradient updates) → **val** (checkpoint selection) → **test** (final honest evaluation, never used for model selection).

In [1]:
# ── Configuration ─────────────────────────────────────────────────────────────
TRAINING_DIR   = 'Training Data'   # directory containing .pt files
CHECKPOINT_DIR = 'checkpoints'     # best model saved here

# Balancing: negative (no-lick) chunks per positive (has-lick) chunk.
# Positive chunks already contain no-lick sections, so fewer pure negatives
# are needed. Set to None to keep all available negatives.
NEG_TO_POS_RATIO = 1.0

VAL_SPLIT  = 0.15   # fraction held out for validation (hyperparameter tuning)
TEST_SPLIT = 0.15   # fraction held out for final evaluation (never touch until done)
BATCH_SIZE = 16
N_EPOCHS   = 60
LR         = 3e-4
POS_WEIGHT = 25.0   # upweight lick frames in BCE loss
GRAD_CLIP  = 1.0
SEED       = 42

In [2]:
import importlib
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

import lick_classifier as lc
importlib.reload(lc)   # pick up edits without restarting kernel

pt_files = lc.load_pt_files(TRAINING_DIR)
if not pt_files:
    raise RuntimeError('No .pt files found. Run the Save cell in LabelingTool.ipynb first.')

Found 2 .pt file(s): ['raw_data_2025-09-01_10-57-58.pt', 'raw_data_2026-01-20_11-10-18.pt']


In [3]:
# ── Data summary ──────────────────────────────────────────────────────────────
all_samples = []
for f in pt_files:
    d = torch.load(str(f), weights_only=False)
    all_samples.extend(d['samples'])

n_pos         = sum(1 for s in all_samples if len(s['lick_times']) > 0)
n_neg         = len(all_samples) - n_pos
n_lick_events = sum(len(s['lick_times']) for s in all_samples)
print(f'Total chunks:        {len(all_samples)}')
print(f'  With licks:        {n_pos}')
print(f'  Without licks:     {n_neg}')
print(f'Total lick events:   {n_lick_events}')
print(f'Avg licks/pos chunk: {n_lick_events / max(n_pos, 1):.1f}')

Total chunks:        174
  With licks:        44
  Without licks:     130
Total lick events:   1080
Avg licks/pos chunk: 24.5


In [4]:
# ── Build dataloaders (train / val / test) ────────────────────────────────────
train_loader, val_loader, test_loader = lc.build_dataloaders(
    pt_files,
    val_split=VAL_SPLIT,
    test_split=TEST_SPLIT,
    batch_size=BATCH_SIZE,
    neg_to_pos_ratio=NEG_TO_POS_RATIO,
    seed=SEED,
)

Balancing: 44 pos  +  44 neg  (ratio 1.0:1,  86 neg discarded)
Train:  62 samples (32 with licks)
Val:    13 samples (6 with licks)
Test:   13 samples (6 with licks)  ← held out


In [5]:
# ── Build model ───────────────────────────────────────────────────────────────
model, device = lc.build_model()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

LickTransformer: 565,761 parameters | device=mps


In [6]:
# ── Training loop ─────────────────────────────────────────────────────────────
Path(CHECKPOINT_DIR).mkdir(exist_ok=True)

history = {'train_loss': [], 'val_loss': [], 'f1': [], 'precision': [], 'recall': []}
best_f1    = 0.0
best_epoch = 0

for epoch in range(1, N_EPOCHS + 1):
    tr_loss = lc.train_epoch(model, train_loader, optimizer, device,
                              pos_weight=POS_WEIGHT, grad_clip=GRAD_CLIP)
    va_loss = lc.eval_epoch(model, val_loader, device, pos_weight=POS_WEIGHT)
    metrics = lc.eval_detection_metrics(model, val_loader, device)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['f1'].append(metrics['f1'])
    history['precision'].append(metrics['precision'])
    history['recall'].append(metrics['recall'])

    if metrics['f1'] > best_f1:
        best_f1    = metrics['f1']
        best_epoch = epoch
        lc.save_checkpoint(
            model,
            Path(CHECKPOINT_DIR) / 'best.pt',
            epoch=epoch, f1=best_f1,
            neg_to_pos_ratio=NEG_TO_POS_RATIO,
            pos_weight=POS_WEIGHT,
        )

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{N_EPOCHS} | '
              f'train={tr_loss:.4f}  val={va_loss:.4f} | '
              f'P={metrics["precision"]:.3f}  R={metrics["recall"]:.3f}  '
              f'F1={metrics["f1"]:.3f}'
              + ('  ← best' if epoch == best_epoch else ''))

print(f'\nBest F1={best_f1:.3f} at epoch {best_epoch}  →  {CHECKPOINT_DIR}/best.pt')

/Users/christopher/.pyenv/versions/cliqr-network-training/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch   1/60 | train=1.0316  val=1.3065 | P=0.111  R=0.048  F1=0.067  ← best


RuntimeError: MPS backend out of memory (MPS allocated: 8.02 GiB, other allocations: 608.59 MiB, max allowed: 9.07 GiB). Tried to allocate 549.32 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
epochs = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, history['train_loss'], label='train')
ax1.plot(epochs, history['val_loss'],   label='val')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss')
ax1.legend()

ax2.plot(epochs, history['f1'],        label='F1')
ax2.plot(epochs, history['precision'], label='Precision', linestyle='--')
ax2.plot(epochs, history['recall'],    label='Recall',    linestyle=':')
ax2.axvline(best_epoch, color='gray', linewidth=0.8, linestyle='--',
            label=f'best (ep {best_epoch})')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Score')
ax2.set_title('Detection metrics (val)')
ax2.legend()

fig.tight_layout()
plt.show()

In [ ]:
# ── Final test-set evaluation (run once, after all tuning is done) ────────────
# Load the best checkpoint and evaluate on the held-out test set.
# Do not re-tune hyperparameters based on these numbers.
best_model, meta = lc.load_checkpoint(Path(CHECKPOINT_DIR) / 'best.pt', device=device)
test_metrics = lc.eval_detection_metrics(best_model, test_loader, device)
print(f'Test results  (best checkpoint, epoch {meta["epoch"]})')
print(f'  Precision : {test_metrics["precision"]:.3f}')
print(f'  Recall    : {test_metrics["recall"]:.3f}')
print(f'  F1        : {test_metrics["f1"]:.3f}')
print(f'  TP={test_metrics["tp"]}  FP={test_metrics["fp"]}  FN={test_metrics["fn"]}')